# 00 · From the Cottom et al. (2024) data to two tidy tables

The maps and the income-group chart in the article all come from one study:

> Cottom, J.W., Cook, E. & Velis, C.A. (2024). *A local-to-global emissions inventory of macroplastic pollution.* **Nature** 633, 101–108. https://doi.org/10.1038/s41586-024-07758-6

The authors modelled what happens to plastic in the household ("municipal") waste of more than 50,000 municipalities worldwide for the year 2020, and published everything on Dryad (licence **CC0**, so we may reuse it freely):
https://doi.org/10.5061/dryad.8cz8w9gxb

That download is a 1.4 GB zip file. You don't need all of it. Two small spreadsheets inside the zip are enough, and a copy of both is already in `data/raw/`:

| File | What's in it |
|---|---|
| `SD_03_..._MFA_Outputs_National.xlsx` | results per country (246 countries and territories) |
| `SD_04_..._MFA_Outputs_Global_Regional_Income.xlsx` | results for the world, UN regions and World Bank income groups |

**Want to check this step yourself?** Download the zip from Dryad, then run the (commented-out) cell below. It pulls only those two files out of the zip.

This notebook turns the spreadsheets into two tidy CSV files in `data/processed/`. The other notebooks only use those CSV files.

In [1]:
# Running this in Google Colab? This cell downloads the project first. (On your own computer it does nothing.)
import os, subprocess, sys
GITHUB_REPO = "MauKruisheer/substack"
ARTICLE = "weeks/20260930-where-does-plastic-end-up"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/repo"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", f"https://github.com/{GITHUB_REPO}", "/content/repo"], check=True)
    os.chdir(f"/content/repo/{ARTICLE}/notebooks")

In [2]:
from pathlib import Path
import zipfile

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
OUT = ROOT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)

NATIONAL = RAW / "SD_03_Cottom_et_al_V1.1.0-G-1223_SPOT_MFA_Outputs_National.xlsx"
INCOME = RAW / "SD_04_Cottom_et_al_V1.1.0-G-1223_SPOT_MFA_Outputs_Global_Regional_Income.xlsx"

In [3]:
# Optional: extract the two spreadsheets from the full Dryad download yourself.
# zip_path = Path("~/Downloads/Cottom_et_al_Nature_Published.zip").expanduser()
# with zipfile.ZipFile(zip_path) as z:
#     for member in z.namelist():
#         if member.startswith("Supplementary_Data/SD_03") or member.startswith("Supplementary_Data/SD_04"):
#             target = RAW / Path(member).name
#             target.write_bytes(z.read(member))
#             print("extracted", target.name)

## How the spreadsheets are organised

Each workbook has three data sheets:

* **`01_Processes`**: tonnes of waste flowing through each step of the waste system (generated, collected, burned, ...)
* **`02_Coefficients`**: the rates in between (e.g. collection coverage)
* **`03_Outputs`**: the summary numbers we need, such as *plastic emissions* in tonnes per year

Each has a `..._legend` sheet explaining the column codes. Let's peek at the legend for the outputs:

In [4]:
legend = pd.read_excel(NATIONAL, sheet_name="03a_Outputs_legend")
legend[legend["Output_ID"].str.startswith("plas", na=False)][["Output_ID", "Output_long_name", "Unit"]]

,Output_ID,Output_long_name,Unit
21,plas_debris_em,Plastic debris emissions,t·y-1
22,plas_burn_em,Openly burned plastic emissions,t·y-1
23,plas_em,Plastic emissions (debris + openly burned),t·y-1
24,plas_debris_em_pct_em,Plastic debris emissions as percentage of plas...,% plastic emissions
25,plas_burn_em_pct_em,Openly burnt plastic emissions as percentage o...,% plastic emissions
26,plas_debris_em_per_cap,Per capita plastic debris emissions,kg·cap-1·y-1
27,plas_burn_em_per_cap,Per capita openly burned plastic,kg·cap-1·y-1
28,plas_em_per_cap,Per capita plastic emissions (debris + openly ...,kg·cap-1·y-1
37,plas_litter_em,Plastic emissions from litter,t·y-1
38,plas_uncol_em,Plastic emissions from uncollected waste,t·y-1


Two things to know before we continue:

1. **"Plastic emissions" = plastic that leaves the managed waste system into the open environment.** This is the article's *waste pollution*. It comes in two forms:
   * **debris**: solid pieces left on land or in water
   * **open burned**: plastic burned in the open (backyards, dumpsites), which ends up in the air as smoke and toxic fumes
2. The model is **probabilistic**: every number was simulated 5,000 times. So each country appears in six rows, one per statistic: `Mean`, `5th percentile`, `Lower quartile`, `Median`, `Upper quartile`, `95th percentile`. We use the **mean** as the headline value and the **5th–95th percentile** as the uncertainty range. (Means add up nicely: the country means sum exactly to the world mean.)

In [5]:
nat = pd.read_excel(NATIONAL, sheet_name="03_Outputs", header=1)
print(nat.shape)
nat[["Statistic", "Country", "ISO3", "Income_cat", "Population_2020", "plas_em", "plas_em_per_cap"]].head(6)

(1476, 69)


,Statistic,Country,ISO3,Income_cat,Population_2020,plas_em,plas_em_per_cap
0,Mean,Aruba,ABW,HIC,106476.813465,67.651361,0.635362
1,5th percentile,Aruba,ABW,HIC,106476.813465,3.954339,0.037138
2,Lower quartile,Aruba,ABW,HIC,106476.813465,7.769187,0.072966
3,Median,Aruba,ABW,HIC,106476.813465,12.591921,0.118260
4,Upper quartile,Aruba,ABW,HIC,106476.813465,63.845453,0.599618
5,95th percentile,Aruba,ABW,HIC,106476.813465,327.813886,3.078735


## Table 1: plastic pollution per country

We keep the mean, 5th and 95th percentile for total and per-capita emissions, and the mean for the parts they're made of.

In [6]:
ID_COLS = ["Country", "ISO3", "Income_cat", "Region", "Subregion", "Population_2020"]

mean = nat[nat["Statistic"] == "Mean"].set_index("ISO3")
p05 = nat[nat["Statistic"] == "5th percentile"].set_index("ISO3")
p95 = nat[nat["Statistic"] == "95th percentile"].set_index("ISO3")

countries = pd.DataFrame({
    "country": mean["Country"],
    "income_group": mean["Income_cat"],
    "region": mean["Region"],
    "subregion": mean["Subregion"],
    "population_2020": mean["Population_2020"].round(0).astype("int64"),
    # total plastic pollution, tonnes per year
    "plastic_pollution_t": mean["plas_em"],
    "plastic_pollution_t_p05": p05["plas_em"],
    "plastic_pollution_t_p95": p95["plas_em"],
    # per person, kg per year
    "plastic_pollution_kg_per_cap": mean["plas_em_per_cap"],
    "plastic_pollution_kg_per_cap_p05": p05["plas_em_per_cap"],
    "plastic_pollution_kg_per_cap_p95": p95["plas_em_per_cap"],
    # the two forms
    "debris_t": mean["plas_debris_em"],
    "open_burned_t": mean["plas_burn_em"],
    # where in the waste system it escapes
    "src_uncollected_t": mean["plas_uncol_em"],
    "src_litter_t": mean["plas_litter_em"],
    "src_collection_t": mean["plas_collection_em"],
    "src_disposal_t": mean["plas_disp_em"],
    "src_sorting_rejects_t": mean["plas_recy_em"],
}).reset_index().rename(columns={"ISO3": "iso3"})

countries = countries.sort_values("plastic_pollution_t", ascending=False)
countries.to_csv(OUT / "cottom2024_country_pollution.csv", index=False, float_format="%.4g")
countries.head(10)

,iso3,country,income_group,region,subregion,population_2020,plastic_pollution_t,plastic_pollution_t_p05,plastic_pollution_t_p95,plastic_pollution_kg_per_cap,plastic_pollution_kg_per_cap_p05,plastic_pollution_kg_per_cap_p95,debris_t,open_burned_t,src_uncollected_t,src_litter_t,src_collection_t,src_disposal_t,src_sorting_rejects_t
100,IND,India,LMC,Asia,Southern Asia,1396453010,9.275777e+06,6.508591e+06,1.267783e+07,6.642384,4.660802,9.078597,3.490204e+06,5.785573e+06,4.961594e+06,312576.257441,171937.124617,3.614131e+06,215537.973520
158,NGA,Nigeria,LMC,Africa,Sub-Saharan Africa,208261372,3.532479e+06,2.645344e+06,4.575801e+06,16.961759,12.702039,21.971433,1.815510e+06,1.716970e+06,2.957211e+06,70290.196054,21108.824889,4.144542e+05,69415.184411
98,IDN,Indonesia,LMC,Asia,South-eastern Asia,271016047,3.352229e+06,2.503945e+06,4.288751e+06,12.369116,9.239101,15.824714,1.409487e+06,1.942742e+06,2.201822e+06,72938.057886,39941.030721,9.830457e+05,54481.508634
40,CHN,China,UMC,Asia,Eastern Asia,1423731552,2.808179e+06,2.102578e+06,3.709376e+06,1.972408,1.476808,2.605390,1.168421e+06,1.639758e+06,2.378708e+06,182158.682775,164467.056694,1.902298e+04,63822.432340
167,PAK,Pakistan,LMC,Asia,Southern Asia,227180827,2.567461e+06,1.785847e+06,3.519867e+06,11.301396,7.860905,15.493682,1.253921e+06,1.313540e+06,1.942542e+06,70356.719818,24945.941203,4.528849e+05,76731.175083
20,BGD,Bangladesh,LMC,Asia,Southern Asia,167132312,1.748215e+06,1.135441e+06,2.446435e+06,10.460064,6.793665,14.637714,8.712382e+05,8.769764e+05,1.324603e+06,53267.424330,17405.941624,2.920789e+05,60859.568990
184,RUS,Russia,UMC,Europe,Eastern Europe,145353329,1.702453e+06,1.133254e+06,2.350681e+06,11.712518,7.796546,16.172188,2.801890e+05,1.422265e+06,6.289870e+05,32798.307550,24514.442429,1.006859e+06,9294.332704
30,BRA,Brazil,UMC,Americas,Latin America and the Caribbean,213090622,1.444824e+06,9.025090e+05,2.052069e+06,6.780325,4.235330,9.630031,4.036140e+05,1.041210e+06,1.026303e+06,38146.688712,23839.271464,3.447431e+05,11791.817639
212,THA,Thailand,UMC,Asia,South-eastern Asia,71462588,9.957185e+05,6.531212e+05,1.378748e+06,13.933423,9.139344,19.293288,1.964242e+05,7.992943e+05,5.468499e+05,23301.997247,14431.939078,4.059660e+05,5168.640496
43,COD,Democratic Republic of the Congo,LIC,Africa,Sub-Saharan Africa,92925478,9.633278e+05,5.736947e+05,1.402377e+06,10.366670,6.173707,15.091412,6.340989e+05,3.292289e+05,7.888870e+05,56605.233741,8053.558732,7.437385e+04,35408.195727


**Sanity check:** the countries should add up to the global total reported in the paper (≈ 52.1 million tonnes, Mt).

In [7]:
print(f"World total: {countries['plastic_pollution_t'].sum() / 1e6:.1f} Mt per year")
print(f"World population: {countries['population_2020'].sum() / 1e9:.2f} billion")
print(f"Per person: {countries['plastic_pollution_t'].sum() * 1000 / countries['population_2020'].sum():.1f} kg per year")

World total: 52.1 Mt per year
World population: 7.83 billion
Per person: 6.7 kg per year


## Table 2: sources of plastic pollution per income group

For the income-group chart we need the emissions split by *where* in the waste system the plastic escapes. These live in the `01_Processes` sheet, so we add up the relevant process codes (see the `01a_Processes_legend` sheet):

| Source | Debris | Open burned |
|---|---|---|
| **Uncollected waste**: no waste collection at all | `P20a` | `P21a` |
| **Litter**: dropped although there is collection | `P2a` | – |
| **Collection system**: lost from bins and trucks | `P6a` | – |
| **Disposal**: escapes from or is burned at dumpsites | `P19a` | `P17a` |
| **Sorting rejects**: thrown out during (mostly informal) recycling | `P31aa, P31ab, P33aa, P33ab` | `P30aa, P30ab, P32aa, P32ab` |

In [8]:
proc = pd.read_excel(INCOME, sheet_name="01_Processes", header=2)
out_inc = pd.read_excel(INCOME, sheet_name="03_Outputs", header=1)

GROUPS = {"World": "World", "HIC": "High income", "UMC": "Upper-middle income",
          "LMC": "Lower-middle income", "LIC": "Low income"}

SOURCES = {
    ("Uncollected waste", "debris"): ["P20a"],
    ("Uncollected waste", "open burned"): ["P21a"],
    ("Litter", "debris"): ["P2a"],
    ("Collection system", "debris"): ["P6a"],
    ("Disposal", "debris"): ["P19a"],
    ("Disposal", "open burned"): ["P17a"],
    ("Sorting rejects", "debris"): ["P31aa", "P31ab", "P33aa", "P33ab"],
    ("Sorting rejects", "open burned"): ["P30aa", "P30ab", "P32aa", "P32ab"],
}

m = proc[(proc["Statistic"] == "Mean") & proc["Aggregation_lv2"].isin(GROUPS)].set_index("Aggregation_lv2")
pop = out_inc[(out_inc["Statistic"] == "Mean") & out_inc["Aggregation_lv2"].isin(GROUPS)].set_index("Aggregation_lv2")["Population_2020"]

rows = []
for code, name in GROUPS.items():
    for (source, form), cols in SOURCES.items():
        tonnes = m.loc[code, cols].sum()
        rows.append({
            "income_code": code, "income_group": name, "source": source, "form": form,
            "tonnes": tonnes,
            "kg_per_cap": tonnes * 1000 / pop[code],
            "population_2020": round(pop[code]),
        })
income = pd.DataFrame(rows)
income.to_csv(OUT / "cottom2024_income_sources.csv", index=False, float_format="%.5g")

# check: the parts add up to the total plastic emissions in 03_Outputs
total_check = out_inc[(out_inc["Statistic"] == "Mean") & out_inc["Aggregation_lv2"].isin(GROUPS)].set_index("Aggregation_lv2")["plas_em"]
check = pd.DataFrame({"sum_of_parts_Mt": income.groupby("income_code")["tonnes"].sum() / 1e6,
                      "reported_total_Mt": total_check / 1e6})
check.round(3)

,sum_of_parts_Mt,reported_total_Mt
HIC,0.163,0.163
LIC,7.489,7.489
LMC,30.409,30.409
UMC,14.046,14.046
World,52.106,52.106


In [9]:
(income.pivot_table(index="income_group", columns="source", values="kg_per_cap", aggfunc="sum")
       .loc[list(GROUPS.values())].round(2))

source,Collection system,Disposal,Litter,Sorting rejects,Uncollected waste
income_group,,,,,
World,0.12,1.63,0.23,0.13,4.54
High income,0.04,0.00,0.06,0.00,0.03
Upper-middle income,0.13,1.27,0.16,0.06,3.57
Lower-middle income,0.14,2.69,0.28,0.20,6.27
Low income,0.12,1.11,0.54,0.31,8.16


Done. The next notebooks read these two files:

* `data/processed/cottom2024_country_pollution.csv`
* `data/processed/cottom2024_income_sources.csv`